# 07. Content Understanding Skill + Image Serving

이 노트북에서는:
1. **Content Understanding Skill** (`ContentUnderstandingSkill`)을 사용하여 PDF/PPTX를 인덱싱합니다.
2. 기존 DI Layout 기반 파이프라인(B-1, B-3)과 CU Skill 파이프라인(B-5, B-6)의 검색 결과를 비교합니다.
3. **Image Serving** — Agentic Retrieval 시 원본 이미지를 직접 반환하는 기능을 테스트합니다.

## Content Understanding Skill이란?

| 항목 | 설명 |
|------|------|
| OData 타입 | `#Microsoft.Skills.Util.ContentUnderstandingSkill` |
| 역할 | DI Layout + 청킹 + AI 이미지 설명을 **단일 스킬**로 통합 |
| 청킹 | Semantic chunking (단락/제목 경계 준수, 고정 크기 X) |
| 이미지 | AI image description (이미지를 텍스트로 설명) |
| API 버전 | `2026-04-01` GA (기본), `2026-05-01-preview` (semantic chunking + image desc) |
| Function App | **불필요** — Native Skill이므로 `func-skills` 미사용 |

## 파이프라인 구성 (B-5, B-6)

| 파이프라인 | 인덱서 | 스킬 체인 | Function App |
|-----------|--------|-----------|-------------|
| **B-5 CU PDF** | `multimodal-cu-indexer-pdf` | CU Skill (semantic chunking + image desc) → Embedding → Index | ❌ 불필요 |
| **B-6 CU PPTX** | `multimodal-cu-indexer-pptx` | CU Skill (semantic chunking + image desc) → Embedding → Index | ❌ 불필요 |

## 사전 요구사항
- `05-multimodal-indexing.ipynb` 실행 완료 (PDF/PPTX가 Blob Storage에 업로드됨)
- `.env` 파일 생성 완료

## 데이터 흐름
```
[Blob Storage — raw-documents/raw/pdf/, raw/pptx/]
    │  AI Search Indexer (Content Understanding Skill)
    ├───────────────────────┐
    ▼ [B-5] CU PDF          ▼ [B-6] CU PPTX
    │                       │
    ▼                       ▼
[multimodal-cu-index-pdf]  [multimodal-cu-index-pptx]
```

## §1. 환경 설정

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

import requests
from azure.identity import DefaultAzureCredential
from openai import AzureOpenAI

# src.pipeline 모듈 임포트
from src.pipeline.indexer_ops import (
    SearchAdminClient,
    run_indexer as _pipeline_run_indexer,
    poll_indexer as _pipeline_poll_indexer,
    get_indexer_status as _pipeline_get_indexer_status,
)

# 필수 환경변수 확인
required_vars = [
    'AZURE_SEARCH_SERVICE_ENDPOINT',
    'AZURE_STORAGE_ACCOUNT_NAME',
    'AZURE_STORAGE_RESOURCE_ID',
    'AZURE_OPENAI_ENDPOINT',
]
for var in required_vars:
    assert os.environ.get(var), f'{var} 환경변수가 설정되지 않았습니다.'

STORAGE_NAME = os.environ['AZURE_STORAGE_ACCOUNT_NAME']
SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
STORAGE_RESOURCE_ID = os.environ['AZURE_STORAGE_RESOURCE_ID']
EMBEDDING_DEPLOYMENT = os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large')
GPT_DEPLOYMENT = os.environ.get('AZURE_OPENAI_GPT54_DEPLOYMENT', 'gpt-5.4')

# ★ CU Skill은 2026-05-01-preview API 버전 필요
API_VERSION = '2026-05-01-preview'

# SearchAdminClient (노트북 전역)
_client = SearchAdminClient()

credential = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
)

# B-5, B-6 인덱서/인덱스 이름
CU_PDF_INDEXER = 'multimodal-cu-indexer-pdf'         # [B-5] CU PDF
CU_PPTX_INDEXER = 'multimodal-cu-indexer-pptx'       # [B-6] CU PPTX
CU_PDF_INDEX = 'multimodal-cu-index-pdf'
CU_PPTX_INDEX = 'multimodal-cu-index-pptx'
CU_PDF_SKILLSET = 'multimodal-cu-skillset-pdf'
CU_PPTX_SKILLSET = 'multimodal-cu-skillset-pptx'
CU_PDF_DATASOURCE = 'multimodal-cu-datasource-pdf'
CU_PPTX_DATASOURCE = 'multimodal-cu-datasource-pptx'

# 기존 파이프라인 인덱스 (비교용)
BASIC_PDF_INDEX = 'multimodal-basic-index-pdf'           # [B-1]
VERBALIZED_PDF_INDEX = 'multimodal-verbalized-index-pdf'  # [B-3]

# Blob 경로
CONTAINER_NAME = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
BLOB_PREFIX_PDF = 'raw/pdf/'
BLOB_PREFIX_PPTX = 'raw/pptx/'

# AI Services (CU Skill용)
AI_SERVICES_SUBDOMAIN = os.environ.get('AZURE_AI_SERVICES_SUBDOMAIN', '')

print('환경 설정 완료')
print(f'  Storage      : {STORAGE_NAME}')
print(f'  AI Search    : {SEARCH_ENDPOINT}')
print(f'  OpenAI       : {OPENAI_ENDPOINT}')
print(f'  Embedding    : {EMBEDDING_DEPLOYMENT}')
print(f'  API Version  : {API_VERSION}')
print(f'  Container    : {CONTAINER_NAME}')
print(f'  B-5 PDF      : {CU_PDF_INDEX} / {CU_PDF_INDEXER}')
print(f'  B-6 PPTX     : {CU_PPTX_INDEX} / {CU_PPTX_INDEXER}')

In [ ]:
# ── REST 헬퍼 함수 ─────────────────────────────────────────

def get_search_headers():
    return _client._headers()


def search_request(method: str, path: str, body: dict | None = None) -> requests.Response:
    """AI Search REST API 호출 (CU API 버전 사용)"""
    sep = '&' if '?' in path else '?'
    url = f'{SEARCH_ENDPOINT}{path}{sep}api-version={API_VERSION}'
    resp = requests.request(method, url, headers=get_search_headers(), json=body, timeout=120)
    return resp


def upsert_resource(path: str, body: dict, label: str = '') -> None:
    """PUT으로 리소스 생성/업데이트"""
    resp = search_request('PUT', path, body)
    if resp.status_code in (200, 201):
        print(f'  ✓ {label or path}')
    else:
        print(f'  ✗ {label or path} → {resp.status_code}')
        print(resp.text[:500])
        resp.raise_for_status()


def delete_if_exists(path: str) -> None:
    resp = search_request('DELETE', path)
    if resp.status_code in (200, 202, 204):
        print(f'  - deleted {path}')


def run_indexer(indexer_name: str) -> bool:
    try:
        resp = search_request('POST', f'/indexers/{indexer_name}/run')
        if resp.status_code in (202, 204):
            print(f'  → Indexer \'{indexer_name}\' 실행 요청')
            return True
        print(f'  ✗ run {indexer_name} → {resp.status_code}: {resp.text[:200]}')
        return False
    except Exception as e:
        print(f'  ✗ {indexer_name} 실행 실패: {e}')
        return False


def get_indexer_status(indexer_name: str) -> dict:
    resp = search_request('GET', f'/indexers/{indexer_name}/status')
    return resp.json() if resp.status_code == 200 else {}


def wait_indexer(indexer_name: str, timeout_sec: int = 1800, poll_interval: int = 15):
    """인덱서 완료 대기."""
    baseline = (get_indexer_status(indexer_name).get('lastResult') or {}).get('startTime')
    start = time.time()
    print(f'  ⏳ {indexer_name} 완료 대기 중...')
    while True:
        status = get_indexer_status(indexer_name)
        top_state = status.get('status', 'unknown')
        last = status.get('lastResult') or {}
        last_state = last.get('status', 'unknown')
        last_start = last.get('startTime')
        processed = last.get('itemsProcessed', 0)
        failed = last.get('itemsFailed', 0)
        elapsed = int(time.time() - start)
        is_new = (last_start is not None and last_start != baseline)
        print(f'    [{elapsed:>4d}s] top={top_state} last={last_state} '
              f'processed={processed} failed={failed}')
        if is_new and last_state in ('success', 'transientFailure', 'persistentFailure') \
                and top_state != 'running':
            return last_state, last
        if elapsed > timeout_sec:
            print(f'    ⚠ timeout ({timeout_sec}s)')
            return 'timeout', last
        time.sleep(poll_interval)


print('REST 헬퍼 함수 준비 완료')

## §2. B-5 Content Understanding PDF 파이프라인

CU Skill 하나로 DI Layout + 시맨틱 청킹 + AI 이미지 설명을 통합 처리합니다.

```
[Blob PDF] → ContentUnderstandingSkill → AzureOpenAIEmbeddingSkill → multimodal-cu-index-pdf
                 ├─ semantic chunking (단락/제목 경계)
                 └─ AI image description
```

In [ ]:
# ── 2a. 인덱스 생성 ─────────────────────────────────────────

def create_cu_index(index_name: str, dimensions: int = 3072) -> None:
    """CU Skill용 인덱스 생성 (B-1과 동일 스키마, 이름만 다름)"""
    print(f'  [index] {index_name}')
    delete_if_exists(f'/indexes/{index_name}')
    upsert_resource(f'/indexes/{index_name}', {
        'name': index_name,
        'fields': [
            {'name': 'id', 'type': 'Edm.String', 'key': True, 'filterable': True, 'analyzer': 'keyword'},
            {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
            {'name': 'source_file_name', 'type': 'Edm.String', 'searchable': True, 'filterable': True},
            {'name': 'source_blob_path', 'type': 'Edm.String', 'filterable': True, 'retrievable': True},
            {'name': 'content_type', 'type': 'Edm.String', 'filterable': True, 'facetable': True},
            {'name': 'file_type', 'type': 'Edm.String', 'filterable': True, 'facetable': True, 'retrievable': True},
            {'name': 'source_category', 'type': 'Edm.String', 'filterable': True, 'facetable': True, 'retrievable': True},
            {'name': 'content', 'type': 'Edm.String', 'searchable': True, 'analyzer': 'ko.lucene'},
            {
                'name': 'content_vector', 'type': 'Collection(Edm.Single)',
                'searchable': True, 'retrievable': False,
                'dimensions': dimensions, 'vectorSearchProfile': 'mm-vector-profile',
            },
            {'name': 'metadata_storage_last_modified', 'type': 'Edm.DateTimeOffset', 'filterable': True, 'sortable': True},
        ],
        'vectorSearch': {
            'profiles': [{'name': 'mm-vector-profile', 'algorithm': 'mm-hnsw'}],
            'algorithms': [{'name': 'mm-hnsw', 'kind': 'hnsw', 'hnswParameters': {'metric': 'cosine'}}],
        },
        'semantic': {
            'configurations': [{
                'name': 'mm-semantic-config',
                'prioritizedFields': {
                    'titleField': {'fieldName': 'source_file_name'},
                    'prioritizedContentFields': [{'fieldName': 'content'}],
                },
            }],
        },
    }, label=f'index {index_name}')


create_cu_index(CU_PDF_INDEX)
print(f'\n✅ B-5 인덱스 생성 완료: {CU_PDF_INDEX}')

In [ ]:
# ── 2b. CU Skillset 생성 ─────────────────────────────────────

def create_cu_skillset(
    skillset_name: str,
    index_name: str,
    openai_endpoint: str,
    embedding_deployment: str,
    dimensions: int = 3072,
    ai_services_subdomain: str = '',
) -> None:
    """Content Understanding Skill + Embedding 스킬셋 생성."""
    print(f'  [skillset] {skillset_name}')

    skillset_payload = {
        'name': skillset_name,
        'description': 'Content Understanding Skill: semantic chunking + AI image description + Embedding',
        'skills': [
            {
                '@odata.type': '#Microsoft.Skills.Util.ContentUnderstandingSkill',
                'name': 'cu-skill',
                'context': '/document',
                'outputMode': 'oneToMany',
                'inputs': [
                    {'name': 'file_data', 'source': '/document/file_data'},
                ],
                'outputs': [
                    {'name': 'chunks', 'targetName': 'cu_chunks'},
                    {'name': 'images', 'targetName': 'cu_images'},
                ],
                'analyzerMode': 'document',
                'analysisOptions': {
                    'chunkingOptions': {
                        'chunkingStrategy': 'semantic',
                        'maxChunkSize': 2000,
                    },
                    'imageOptions': {
                        'returnImages': True,
                        'imageResolution': 'medium',
                    },
                },
            },
            {
                '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
                'name': 'embedding-skill',
                'context': '/document/cu_chunks/*',
                'resourceUri': openai_endpoint,
                'deploymentId': embedding_deployment,
                'modelName': 'text-embedding-3-large',
                'dimensions': dimensions,
                'inputs': [{'name': 'text', 'source': '/document/cu_chunks/*'}],
                'outputs': [{'name': 'embedding', 'targetName': 'chunk_vector'}],
            },
        ],
        'indexProjections': {
            'selectors': [{
                'targetIndexName': index_name,
                'parentKeyFieldName': 'parent_id',
                'sourceContext': '/document/cu_chunks/*',
                'mappings': [
                    {'name': 'content', 'source': '/document/cu_chunks/*'},
                    {'name': 'content_vector', 'source': '/document/cu_chunks/*/chunk_vector'},
                    {'name': 'source_file_name', 'source': '/document/metadata_storage_name'},
                    {'name': 'source_blob_path', 'source': '/document/metadata_storage_path'},
                    {'name': 'metadata_storage_last_modified', 'source': '/document/metadata_storage_last_modified'},
                ],
            }],
            'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
        },
    }

    if ai_services_subdomain:
        skillset_payload['cognitiveServices'] = {
            '@odata.type': '#Microsoft.Azure.Search.AIServicesByIdentity',
            'subdomainUrl': ai_services_subdomain.rstrip('/') + '/',
            'identity': None,
        }

    upsert_resource(f'/skillsets/{skillset_name}', skillset_payload, label=f'skillset {skillset_name}')


create_cu_skillset(
    skillset_name=CU_PDF_SKILLSET,
    index_name=CU_PDF_INDEX,
    openai_endpoint=OPENAI_ENDPOINT,
    embedding_deployment=EMBEDDING_DEPLOYMENT,
    ai_services_subdomain=AI_SERVICES_SUBDOMAIN,
)
print(f'\n✅ B-5 스킬셋 생성 완료: {CU_PDF_SKILLSET}')

In [ ]:
# ── 2c. 데이터소스 생성 ──────────────────────────────────────

def create_cu_datasource(datasource_name: str, prefix: str) -> None:
    """Blob 데이터소스 생성 (CU 파이프라인용)"""
    print(f'  [datasource] {datasource_name}')
    delete_if_exists(f'/datasources/{datasource_name}')
    upsert_resource(f'/datasources/{datasource_name}', {
        'name': datasource_name,
        'type': 'azureblob',
        'credentials': {'connectionString': f'ResourceId={STORAGE_RESOURCE_ID}'},
        'container': {'name': CONTAINER_NAME, 'query': prefix},
        'dataChangeDetectionPolicy': {
            '@odata.type': '#Microsoft.Azure.Search.HighWaterMarkChangeDetectionPolicy',
            'highWaterMarkColumnName': 'metadata_storage_last_modified',
        },
    }, label=f'datasource {datasource_name}')


create_cu_datasource(CU_PDF_DATASOURCE, BLOB_PREFIX_PDF)
print(f'\n✅ B-5 데이터소스 생성 완료: {CU_PDF_DATASOURCE}')

In [ ]:
# ── 2d. 인덱서 생성 ──────────────────────────────────────────

def create_cu_indexer(
    indexer_name: str,
    datasource_name: str,
    index_name: str,
    skillset_name: str,
    indexed_extensions: str | None = None,
) -> None:
    """CU 파이프라인 인덱서 생성."""
    print(f'  [indexer] {indexer_name}')

    config: dict = {
        'dataToExtract': 'contentAndMetadata',
        'parsingMode': 'default',
        'allowSkillsetToReadFileData': True,
    }
    if indexed_extensions:
        config['indexedFileNameExtensions'] = indexed_extensions

    indexer_payload = {
        'name': indexer_name,
        'dataSourceName': datasource_name,
        'targetIndexName': index_name,
        'skillsetName': skillset_name,
        'parameters': {
            'batchSize': 10,
            'maxFailedItems': 20,
            'maxFailedItemsPerBatch': 10,
            'configuration': config,
        },
        'fieldMappings': [
            {'sourceFieldName': 'metadata_storage_path', 'targetFieldName': 'source_blob_path'},
            {'sourceFieldName': 'metadata_storage_name', 'targetFieldName': 'source_file_name'},
        ],
    }

    upsert_resource(f'/indexers/{indexer_name}', indexer_payload, label=f'indexer {indexer_name}')


create_cu_indexer(
    indexer_name=CU_PDF_INDEXER,
    datasource_name=CU_PDF_DATASOURCE,
    index_name=CU_PDF_INDEX,
    skillset_name=CU_PDF_SKILLSET,
    indexed_extensions='.pdf',
)
print(f'\n✅ B-5 인덱서 생성 완료: {CU_PDF_INDEXER}')

In [ ]:
# ── 2e. B-5 인덱서 실행 + 완료 대기 ─────────────────────────
print('='*60)
print('[B-5] Content Understanding PDF 인덱서 실행')
print('    CU Skill (semantic chunking + image desc) → Embedding')
print('='*60)

if run_indexer(CU_PDF_INDEXER):
    time.sleep(5)
    state, result = wait_indexer(CU_PDF_INDEXER, timeout_sec=1800)
    print(f'\n  결과: {state}')
    if result:
        print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
        if result.get('errors'):
            for err in result['errors'][:3]:
                print(f'    ERROR: {err.get("message", "")[:200]}')

## §3. B-6 Content Understanding PPTX 파이프라인

B-5와 동일한 CU Skill 구성을 PPTX에 적용합니다.

```
[Blob PPTX] → ContentUnderstandingSkill → AzureOpenAIEmbeddingSkill → multimodal-cu-index-pptx
```

In [ ]:
# ── 3a. B-6 인덱스 + 스킬셋 + 데이터소스 + 인덱서 생성 ──────
print('='*60)
print('[B-6] Content Understanding PPTX 파이프라인 생성')
print('='*60)

# 인덱스
create_cu_index(CU_PPTX_INDEX)

# 스킬셋
create_cu_skillset(
    skillset_name=CU_PPTX_SKILLSET,
    index_name=CU_PPTX_INDEX,
    openai_endpoint=OPENAI_ENDPOINT,
    embedding_deployment=EMBEDDING_DEPLOYMENT,
    ai_services_subdomain=AI_SERVICES_SUBDOMAIN,
)

# 데이터소스
create_cu_datasource(CU_PPTX_DATASOURCE, BLOB_PREFIX_PPTX)

# 인덱서
create_cu_indexer(
    indexer_name=CU_PPTX_INDEXER,
    datasource_name=CU_PPTX_DATASOURCE,
    index_name=CU_PPTX_INDEX,
    skillset_name=CU_PPTX_SKILLSET,
    indexed_extensions='.pptx',
)

print(f'\n✅ B-6 파이프라인 생성 완료')

In [ ]:
# ── 3b. B-6 인덱서 실행 + 완료 대기 ─────────────────────────
print('='*60)
print('[B-6] Content Understanding PPTX 인덱서 실행')
print('    CU Skill (semantic chunking + image desc) → Embedding')
print('='*60)

if run_indexer(CU_PPTX_INDEXER):
    time.sleep(5)
    state, result = wait_indexer(CU_PPTX_INDEXER, timeout_sec=1800)
    print(f'\n  결과: {state}')
    if result:
        print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
        if result.get('errors'):
            for err in result['errors'][:3]:
                print(f'    ERROR: {err.get("message", "")[:200]}')

## §4. 인덱싱 결과 확인

B-5, B-6 인덱스에 저장된 문서 수와 샘플을 확인합니다.

In [ ]:
# ── 4a. 인덱스 통계 ──────────────────────────────────────────

def get_index_stats(index_name: str) -> dict:
    resp = search_request('GET', f'/indexes/{index_name}/stats')
    return resp.json() if resp.status_code == 200 else {}


print(f'{"인덱스":<35} {"문서 수":>10} {"저장 크기":>15}')
print('-'*65)

for idx_name in [CU_PDF_INDEX, CU_PPTX_INDEX]:
    stats = get_index_stats(idx_name)
    doc_count = stats.get('documentCount', 'N/A')
    storage = stats.get('storageSize', 0)
    storage_str = f'{storage:,} bytes' if isinstance(storage, int) else 'N/A'
    print(f'{idx_name:<35} {doc_count:>10} {storage_str:>15}')

In [ ]:
# ── 4b. 샘플 문서 조회 ───────────────────────────────────────

def search_sample(index_name: str, top: int = 3) -> list:
    """인덱스에서 샘플 문서 조회"""
    resp = search_request('POST', f'/indexes/{index_name}/docs/search', {
        'search': '*',
        'top': top,
        'select': 'id,source_file_name,content_type,content',
    })
    if resp.status_code == 200:
        return resp.json().get('value', [])
    return []


for label, idx in [('B-5 CU PDF', CU_PDF_INDEX), ('B-6 CU PPTX', CU_PPTX_INDEX)]:
    print('='*60)
    print(f'[{label}] {idx}')
    print('='*60)
    docs = search_sample(idx)
    if not docs:
        print('  (문서 없음)')
    for doc in docs:
        print(f'\n  📄 {doc.get("source_file_name", "?")} [{doc.get("content_type", "?")}]')
        content = doc.get('content', '') or ''
        print(f'     {content[:300]}...' if len(content) > 300 else f'     {content}')
    print()

## §5. DI Layout vs Content Understanding 검색 비교

동일 쿼리로 3개 파이프라인의 검색 결과를 비교합니다.

| 파이프라인 | 인덱스 | 청킹 방식 | 이미지 처리 |
|-----------|--------|-----------|------------|
| **B-1** Basic PDF | `multimodal-basic-index-pdf` | Custom markdown_split (2000자 고정) | 없음 |
| **B-3** Verbalized PDF | `multimodal-verbalized-index-pdf` | Custom markdown_split (2000자 고정) | GPT-5.4 Vision → 텍스트 변환 |
| **B-5** CU PDF | `multimodal-cu-index-pdf` | Semantic chunking (단락/제목 경계) | AI image description (내장) |

### 주요 비교 포인트
- **청킹**: 고정 크기 vs 시맨틱 (의미 단위 분할)
- **이미지**: 없음 vs GPT Verbalization vs CU AI Description
- **스킬 개수**: B-1은 3개 스킬, B-3은 4개, B-5는 **2개** (CU + Embedding)

In [ ]:
# ── 5a. 하이브리드 검색 비교 함수 ────────────────────────────

def get_embedding(text: str) -> list[float]:
    """텍스트 임베딩 생성"""
    token = credential.get_token('https://cognitiveservices.azure.com/.default').token
    client = AzureOpenAI(
        azure_endpoint=OPENAI_ENDPOINT,
        api_version='2024-12-01-preview',
        azure_ad_token=token,
    )
    response = client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=text,
    )
    return response.data[0].embedding


def hybrid_search(index_name: str, query: str, top: int = 3) -> list[dict]:
    """하이브리드 검색 (키워드 + 벡터 + 시맨틱 랭커)"""
    vector = get_embedding(query)
    body = {
        'search': query,
        'top': top,
        'select': 'id,source_file_name,content',
        'queryType': 'semantic',
        'semanticConfiguration': 'mm-semantic-config',
        'vectorQueries': [{
            'kind': 'vector',
            'vector': vector,
            'fields': 'content_vector',
            'k': top,
        }],
    }
    resp = search_request('POST', f'/indexes/{index_name}/docs/search', body)
    if resp.status_code == 200:
        return resp.json().get('value', [])
    print(f'  ✗ {index_name} 검색 실패: {resp.status_code}')
    return []


print('검색 비교 함수 준비 완료')

In [ ]:
# ── 5b. 3-way 검색 비교 ──────────────────────────────────────

COMPARISON_QUERIES = [
    '차트에서 보여주는 연도별 매출 추이를 설명해주세요',
    '프로젝트 아키텍처 다이어그램에 포함된 구성요소를 알려주세요',
    '주요 결론과 향후 계획을 요약해주세요',
]

INDEXES_TO_COMPARE = [
    ('B-1 Basic PDF', BASIC_PDF_INDEX),
    ('B-3 Verbalized PDF', VERBALIZED_PDF_INDEX),
    ('B-5 CU PDF', CU_PDF_INDEX),
]

for query in COMPARISON_QUERIES:
    print('\n' + '='*80)
    print(f'🔍 쿼리: {query}')
    print('='*80)

    for label, idx_name in INDEXES_TO_COMPARE:
        print(f'\n  ── [{label}] {idx_name} ──')
        results = hybrid_search(idx_name, query, top=2)
        if not results:
            print('    (결과 없음 또는 인덱스 미존재)')
            continue
        for i, doc in enumerate(results, 1):
            score = doc.get('@search.score', 0)
            reranker = doc.get('@search.rerankerScore', 0)
            fname = doc.get('source_file_name', '?')
            content = (doc.get('content', '') or '')[:250]
            print(f'    [{i}] 📄 {fname}  (score={score:.4f}, reranker={reranker:.4f})')
            print(f'        {content}...')

### §5 비교 해석

| 항목 | B-1 Basic | B-3 Verbalized | B-5 CU |
|------|-----------|----------------|--------|
| **청킹** | 고정 2000자 (markdown_split) | 고정 2000자 | 시맨틱 (단락/제목 경계) |
| **이미지 정보** | ❌ 없음 | ✅ GPT-5.4 Vision 텍스트 변환 | ✅ AI image description (내장) |
| **Custom Skill 의존** | ✅ markdown_split | ✅ verbalize + markdown_split | ❌ 불필요 |
| **스킬 수** | 3개 (DI + split + embed) | 4개 (DI + verb + split + embed) | **2개** (CU + embed) |
| **인덱싱 비용** | 낮음 | 높음 (GPT Vision 호출) | 중간 |
| **이미지 질문 응답** | 약함 | 강함 | 강함 |

## §6. Image Serving 설정 및 테스트

**Image Serving**은 Agentic Retrieval 시 문서 내 추출된 이미지를 원본 그대로 반환하는 기능입니다.

- Content Understanding Skill이 인덱싱 시 이미지를 추출·저장
- 검색 시 텍스트와 함께 이미지를 직접 제공
- API: `2026-05-01-preview`

### 설정 단계
1. CU 인덱스를 Foundry IQ Knowledge Source로 등록 (Image Serving 활성화)
2. Knowledge Agent 생성/업데이트
3. Agentic Retrieval 쿼리로 이미지 포함 결과 확인

In [ ]:
# ── 6a. Knowledge Source 등록 (Image Serving 활성화) ──────────

AGENTIC_API_VERSION = '2026-05-01-preview'
AOAI_RESOURCE_URI = OPENAI_ENDPOINT.rstrip('/')
PLANNER_DEPLOY = os.environ.get('AZURE_OPENAI_PLANNER_DEPLOYMENT', 'gpt-4o')

CU_KS_NAME = 'ks-cu-pdf'
CU_AGENT_NAME = 'cu-knowledge-agent'


def _agentic_hdr():
    tok = credential.get_token('https://search.azure.com/.default').token
    return {'Authorization': f'Bearer {tok}', 'Content-Type': 'application/json'}


# Knowledge Source with Image Serving
ks_body = {
    'name': CU_KS_NAME,
    'kind': 'searchIndex',
    'description': (
        'Content Understanding Skill로 인덱싱된 PDF 문서. '
        '시맨틱 청킹 + AI 이미지 설명이 포함된 멀티모달 인덱스. '
        'Image Serving 활성화 — 검색 시 원본 이미지 직접 반환.'
    ),
    'searchIndexParameters': {
        'searchIndexName': CU_PDF_INDEX,
        'sourceDataSelect': 'id,source_file_name,content',
    },
    'indexConfiguration': {
        'contentExtractionMode': 'standard',
        'imageServing': {'enabled': True},
    },
}

url = f'{SEARCH_ENDPOINT}/knowledgesources(\'{CU_KS_NAME}\')?api-version={AGENTIC_API_VERSION}'
r = requests.put(url, headers=_agentic_hdr(), json=ks_body)
print(f'  KS \'{CU_KS_NAME}\' → {r.status_code}')
if r.status_code >= 400:
    print(r.text[:500])

In [ ]:
# ── 6b. Knowledge Agent 생성 ─────────────────────────────────

agent_body = {
    'name': CU_AGENT_NAME,
    'description': 'Content Understanding 기반 멀티모달 검색 + Image Serving',
    'models': [{
        'kind': 'azureOpenAI',
        'azureOpenAIParameters': {
            'resourceUri': AOAI_RESOURCE_URI,
            'deploymentId': PLANNER_DEPLOY,
            'modelName': PLANNER_DEPLOY,
        },
    }],
    'knowledgeSources': [{
        'name': CU_KS_NAME,
        'alwaysQuerySource': True,
        'includeReferences': True,
        'includeReferenceSourceData': True,
        'maxSubQueries': 3,
        'rerankerThreshold': 1.0,
    }],
    'outputConfiguration': {
        'modality': 'answerSynthesis',
        'includeActivity': True,
        'attemptFastPath': False,
    },
    'requestLimits': {'maxRuntimeInSeconds': 60, 'maxOutputSize': 16000},
    'retrievalInstructions': (
        '너는 멀티모달 문서 검색 어시스턴트다. Content Understanding으로 인덱싱된 PDF 문서에서 '
        '텍스트와 이미지를 함께 검색한다. 이미지가 포함된 결과가 있으면 이미지 정보도 함께 제공하라.'
    ),
}

url = f'{SEARCH_ENDPOINT}/agents(\'{CU_AGENT_NAME}\')?api-version={AGENTIC_API_VERSION}'
r = requests.put(url, headers=_agentic_hdr(), json=agent_body)
print(f'  Agent \'{CU_AGENT_NAME}\' → {r.status_code}')
if r.status_code >= 400:
    print(r.text[:500])

In [ ]:
# ── 6c. Agentic Retrieval 쿼리 (Image Serving 포함) ──────────

def agentic_retrieve(agent_name: str, query: str) -> dict:
    """Knowledge Agent를 통한 Agentic Retrieval 쿼리"""
    body = {
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': query}]}],
    }
    url = f'{SEARCH_ENDPOINT}/agents(\'{agent_name}\')/retrieve?api-version={AGENTIC_API_VERSION}'

    delay = 5
    for attempt in range(4):
        r = requests.post(url, headers=_agentic_hdr(), json=body, timeout=120)
        if r.status_code == 429:
            print(f'   ⏳ 429 — {delay}s 대기 ({attempt+1}/4)')
            time.sleep(delay)
            delay *= 2
            continue
        r.raise_for_status()
        return r.json()

    raise RuntimeError('retrieve 429 retries exhausted')


# 이미지 관련 쿼리로 테스트
IMAGE_QUERIES = [
    '차트와 그래프에서 보이는 핵심 데이터를 설명해주세요',
    '아키텍처 다이어그램의 구성요소를 분석해주세요',
]

for query in IMAGE_QUERIES:
    print('\n' + '='*70)
    print(f'🔍 Agentic Retrieval: {query}')
    print('='*70)

    try:
        result = agentic_retrieve(CU_AGENT_NAME, query)

        # 응답 텍스트
        for msg in result.get('response', []):
            for c in msg.get('content', []):
                if c.get('type') == 'text':
                    print(f'\n  📝 응답: {c["text"][:500]}')
                elif c.get('type') == 'image_url':
                    print(f'  🖼️ 이미지 반환됨: {c.get("image_url", {}).get("url", "")[:100]}...')

        # references (이미지 포함 여부 확인)
        refs = result.get('references', [])
        print(f'\n  📚 References: {len(refs)}건')
        for ref in refs[:5]:
            sd = ref.get('sourceData', {})
            fname = sd.get('source_file_name', '?')
            has_image = any(
                c.get('type') == 'image_url'
                for c in ref.get('content', [])
            )
            image_marker = ' 🖼️' if has_image else ''
            print(f'    - {fname}{image_marker}')

        # activity (sub-query trace)
        activities = result.get('activity', [])
        sub_q = sum(1 for a in activities if a.get('type') == 'searchIndex')
        print(f'\n  🔄 Sub-queries: {sub_q}건')

    except Exception as e:
        print(f'  ✗ 에러: {e}')

## §7. Verbalized vs Image Serving 비교

이미지가 포함된 문서 검색 시 두 접근 방식의 차이를 비교합니다.

| 항목 | Verbalized (B-3) | Image Serving (B-5 + CU) |
|------|-----------------|-------------------------|
| **인덱싱 시** | GPT-5.4 Vision이 이미지 → 텍스트 변환 | CU Skill이 이미지 추출 + AI 설명 생성 |
| **검색 시** | 텍스트 설명으로 검색, 원본 이미지 없음 | 텍스트 검색 + 원본 이미지 직접 반환 |
| **모델 reasoning** | 텍스트 기반 추론만 가능 | 원본 이미지 기반 시각 추론 가능 |
| **비용** | GPT Vision 호출 비용 (높음) | CU Skill 비용 (중간) |
| **장점** | 벡터 검색으로 이미지 내용 검색 가능 | 원본 이미지 보존, 시각 정보 손실 없음 |
| **단점** | 이미지 변환 시 정보 손실 가능 | Image Serving API 제한 (preview) |

In [ ]:
# ── 7a. Verbalized vs CU 동일 쿼리 비교 ──────────────────────

VISUAL_QUERY = '문서에 포함된 차트나 다이어그램의 핵심 내용을 설명해주세요'

print('='*80)
print(f'🔍 비교 쿼리: {VISUAL_QUERY}')
print('='*80)

# ── Verbalized (B-3): 텍스트 변환된 이미지 정보로 검색
print('\n── [B-3 Verbalized PDF] — 이미지를 텍스트로 변환하여 검색 ──')
verb_results = hybrid_search(VERBALIZED_PDF_INDEX, VISUAL_QUERY, top=3)
for i, doc in enumerate(verb_results, 1):
    fname = doc.get('source_file_name', '?')
    content = (doc.get('content', '') or '')[:300]
    score = doc.get('@search.rerankerScore', 0)
    print(f'  [{i}] 📄 {fname}  (reranker={score:.4f})')
    print(f'      {content}...')
print('  → 특징: 이미지 내용이 텍스트로 변환되어 검색됨, 원본 이미지 없음')

# ── CU (B-5): 시맨틱 청킹 + AI image description
print('\n── [B-5 CU PDF] — Content Understanding + Image Serving ──')
cu_results = hybrid_search(CU_PDF_INDEX, VISUAL_QUERY, top=3)
for i, doc in enumerate(cu_results, 1):
    fname = doc.get('source_file_name', '?')
    content = (doc.get('content', '') or '')[:300]
    score = doc.get('@search.rerankerScore', 0)
    print(f'  [{i}] 📄 {fname}  (reranker={score:.4f})')
    print(f'      {content}...')
print('  → 특징: 시맨틱 청킹 + Agentic Retrieval 시 Image Serving으로 원본 이미지 반환')

In [ ]:
# ── 7b. Agentic Retrieval로 Image Serving 실제 동작 확인 ──────

print('\n── Agentic Retrieval (Image Serving) 실제 동작 ──')
print(f'   Agent: {CU_AGENT_NAME}')
print(f'   쿼리: {VISUAL_QUERY}')
print()

try:
    result = agentic_retrieve(CU_AGENT_NAME, VISUAL_QUERY)

    text_count = 0
    image_count = 0

    for msg in result.get('response', []):
        for c in msg.get('content', []):
            if c.get('type') == 'text':
                text_count += 1
                print(f'  📝 텍스트 응답:')
                print(f'     {c["text"][:600]}')
            elif c.get('type') == 'image_url':
                image_count += 1
                img_url = c.get('image_url', {}).get('url', '')
                print(f'  🖼️ 이미지 #{image_count}: {img_url[:120]}...')

    print(f'\n  📊 응답 구성: 텍스트 {text_count}개, 이미지 {image_count}개')

    if image_count > 0:
        print('  ✅ Image Serving 동작 확인 — 원본 이미지가 텍스트와 함께 반환됨')
    else:
        print('  ⚠️ 이미지 반환 없음 — 쿼리 결과에 이미지가 포함되지 않았거나 Image Serving 미활성화')

except Exception as e:
    print(f'  ✗ Agentic Retrieval 실패: {e}')
    print('  → Knowledge Agent 또는 Knowledge Source 설정을 확인하세요.')

## §8. 정리

### 전체 멀티모달 파이프라인 비교 (6개 + Image Serving)

| # | 파이프라인 | 인덱서 | 스킬 체인 | Custom Skill | 청킹 | 이미지 |
|---|-----------|--------|-----------|-------------|------|--------|
| B-1 | Basic PDF | `multimodal-basic-indexer-pdf` | DI Layout → split → embed | ✅ markdown_split | 고정 2000자 | ❌ |
| B-2 | Basic PPTX | `multimodal-basic-indexer-pptx` | DI Layout → split → embed | ✅ pptx_page_split | 페이지 단위 | ❌ |
| B-3 | Verbalized PDF | `multimodal-verbalized-indexer-pdf` | DI Layout → verbalize → split → embed | ✅ verbalize + split | 고정 2000자 | GPT Vision 텍스트 변환 |
| B-4 | Verbalized PPTX | `multimodal-verbalized-indexer-pptx` | DI Layout → verbalize → split → embed | ✅ verbalize + split | 고정 2000자 | GPT Vision 텍스트 변환 |
| **B-5** | **CU PDF** | `multimodal-cu-indexer-pdf` | **CU Skill → embed** | ❌ 불필요 | **시맨틱** | **AI image desc** |
| **B-6** | **CU PPTX** | `multimodal-cu-indexer-pptx` | **CU Skill → embed** | ❌ 불필요 | **시맨틱** | **AI image desc** |

### Image Serving 비교

| 항목 | Verbalized (B-3/B-4) | Image Serving (B-5/B-6 + CU) |
|------|---------------------|-----------------------------|
| 이미지 인덱싱 | GPT Vision → 텍스트 변환 후 저장 | CU Skill → 이미지 추출 + AI 설명 저장 |
| 검색 방식 | 텍스트 유사도 검색 | 텍스트 + 원본 이미지 반환 |
| Agentic Retrieval | 텍스트만 반환 | **텍스트 + 이미지 동시 반환** |
| 시각 정보 보존 | 변환 시 손실 가능 | **원본 보존** |
| 비용 | GPT Vision 호출 (높음) | CU Skill (중간) |
| API 요구사항 | `2024-11-01-preview` | `2026-05-01-preview` |

### 노트북 구조

| # | 파일 | 범위 |
|---|------|------|
| 05 | `05-multimodal-indexing.ipynb` | B-1~B-4 (DI Layout 기반) 인덱싱 |
| 06 | `06-multimodal-search.ipynb` | Basic vs Verbalized 검색 비교 → [열기](06-multimodal-search.ipynb) |
| **07** | **`07-content-understanding.ipynb`** | **B-5/B-6 CU Skill + Image Serving + 비교** |